# Curriculum Learning for Mathematical Reasoning: Why Design Matters More Than You Think

**Author:** Timur Khairulov  
**Date:** November 13, 2025  
**Blog Post:** [https://khrtim.github.io/blog/2025/11/curriculum-learning-math-reasoning/](https://khrtim.github.io/blog/2025/11/curriculum-learning-math-reasoning/)

---

## TL;DR

I trained two language models (PHI-2 and SmolLM2) on GSM8K math problems using curriculum learning with two different difficulty estimation methods. **Key finding:** The complexity-based curriculum improved PHI-2 by 2.34% over baseline, while a naive answer-length curriculum actually hurt performance (-0.78%). Small models (135M params) showed minimal improvement, suggesting curriculum learning is most effective for medium-sized models.

## Setup and Dependencies

Install required packages for visualization:

In [ ]:
# Uncomment to install dependencies
# !pip install plotly pandas numpy matplotlib seaborn

In [ ]:
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np

# Set default template
import plotly.io as pio
pio.templates.default = "plotly_white"

## Experimental Data

Results from training PHI-2 (2.7B) and SmolLM2 (135M) on GSM8K dataset with different curriculum methods:

In [ ]:
# Experimental results data
results_data = {
    'Model': ['PHI-2', 'PHI-2', 'PHI-2', 'SmolLM2', 'SmolLM2', 'SmolLM2'],
    'Method': ['Baseline', 'Answer Length', 'Complexity Score', 'Baseline', 'Answer Length', 'Complexity Score'],
    'Exact_Match': [60.16, 59.38, 62.50, 2.15, 2.73, 2.93],
    'Contains_Answer': [69.34, 70.51, 72.27, 20.51, 23.63, 25.20]
}

# Curriculum progression data
progression_data = {
    'PHI-2': {
        'Answer Length': {'Easy': 50.59, 'Normal': 55.86, 'Difficult': 59.38},
        'Complexity Score': {'Easy': 54.10, 'Normal': 59.38, 'Difficult': 62.50},
        'Baseline': 60.16
    },
    'SmolLM2': {
        'Answer Length': {'Easy': 2.54, 'Normal': 3.71, 'Difficult': 2.73},
        'Complexity Score': {'Easy': 3.32, 'Normal': 2.54, 'Difficult': 2.93},
        'Baseline': 2.15
    }
}

# Create DataFrame
df = pd.DataFrame(results_data)
print("\nExperimental Results:")
print(df.to_string(index=False))

---

## Introduction

Large Language Models have shown impressive capabilities in mathematical reasoning, but training them effectively remains a challenge. Inspired by how humans learn math progressively (simple addition before calculus), I investigated whether **curriculum learning** - training models on progressively harder problems - improves mathematical reasoning performance.

### Research Questions
1. Does curriculum learning improve math problem-solving over standard training?
2. How should we estimate problem difficulty for math tasks?
3. Do different model sizes benefit differently from curriculum learning?

## Experimental Setup

### Models Tested
- **PHI-2** (2.7B parameters) - Microsoft's efficient reasoning model
- **SmolLM2** (135M parameters) - A tiny but capable language model

### Dataset
- **GSM8K**: 8.5K grade school math word problems
- Training: 50 samples per curriculum stage (150 total)
- Testing: 512 samples

### Curriculum Methods

#### 1. Answer Length Method (Baseline Curriculum)
Simple heuristic based on solution length:
- **Easy:** Solutions < 168 characters
- **Normal:** Solutions 168-280 characters
- **Difficult:** Solutions > 280 characters

**Assumption:** Longer solutions = harder problems

#### 2. Complexity Score Method (Novel Approach)
Multi-factor difficulty score:
```python
difficulty = solution_steps × operation_complexity

where:
- solution_steps = number of lines in solution
- operation_complexity = weighted count of operations
  (multiplication/division weighted 1.5x vs addition/subtraction)
```

**Assumption:** Problems requiring more steps and complex operations are harder

### Training Details
- **Strategy:** LoRA fine-tuning (memory efficient)
- **Epochs:** 3 per curriculum stage
- **Batch size:** 4 (gradient accumulation: 4)
- **Learning rate:** 3e-4
- **Progression:** Easy → Normal → Difficult (merged models between stages)

---

## Results

### Main Finding: Curriculum Design Matters More Than You Think

In [ ]:
# Performance Heatmap
models = ['PHI-2 (2.7B)', 'SmolLM2 (135M)']
methods = ['Baseline', 'Answer Length', 'Complexity Score']
z_data = [
    [60.16, 59.38, 62.50],  # PHI-2
    [2.15, 2.73, 2.93]       # SmolLM2
]

fig = go.Figure(data=go.Heatmap(
    z=z_data,
    x=methods,
    y=models,
    colorscale='Blues',
    text=[[f"{val:.2f}%" for val in row] for row in z_data],
    texttemplate="%{text}",
    textfont={"size": 14},
    hovertemplate='<b>%{y}</b><br>Method: %{x}<br>Accuracy: %{z:.2f}%<extra></extra>',
    colorbar=dict(title="Accuracy (%)")
))

fig.update_layout(
    title="Performance Heatmap: Model × Training Method",
    xaxis_title="Training Method",
    height=400,
    font=dict(size=12)
)

fig.show()

### PHI-2 (2.7B parameters)

| Method | Exact Match | Improvement |
|--------|------------|-------------|
| Baseline | 60.16% | - |
| Answer Length Curriculum | 59.38% | **-0.78%** ⬇️ |
| Complexity Score Curriculum | 62.50% | **+2.34%** ⬆️ |

**Key Insight:** The *wrong* curriculum can hurt performance! The answer-length method slightly degraded performance, while the complexity-based method improved it.

### SmolLM2 (135M parameters)

| Method | Exact Match | Improvement |
|--------|------------|-------------|
| Baseline | 2.15% | - |
| Answer Length Curriculum | 2.73% | +0.58% |
| Complexity Score Curriculum | 2.93% | +0.78% |

**Key Insight:** Very small models struggle with mathematical reasoning regardless of curriculum (2-3% accuracy). However, curriculum learning still provides modest improvements.

In [ ]:
# Baseline Comparison
fig = go.Figure()

phi2_values = [60.16, 59.38, 62.50]
smollm2_values = [2.15, 2.73, 2.93]

fig.add_trace(go.Bar(
    name='PHI-2 (2.7B)',
    x=methods,
    y=phi2_values,
    text=[f"{v:.2f}%" for v in phi2_values],
    textposition='outside',
    marker_color='#2E86AB'
))

fig.add_trace(go.Bar(
    name='SmolLM2 (135M)',
    x=methods,
    y=smollm2_values,
    text=[f"{v:.2f}%" for v in smollm2_values],
    textposition='outside',
    marker_color='#A23B72'
))

fig.update_layout(
    title='Baseline vs Curriculum Learning Performance',
    xaxis_title='Training Method',
    yaxis_title='Exact Match Accuracy (%)',
    barmode='group',
    height=500,
    showlegend=True
)

fig.show()

---

## Deep Dive: What Makes a Good Curriculum?

### Stage-by-Stage Progression

In [ ]:
# Curriculum Progression for PHI-2
stages = ['Easy', 'Normal', 'Difficult']

fig = go.Figure()

# Answer Length progression
al_values = [50.59, 55.86, 59.38]
fig.add_trace(go.Scatter(
    x=stages,
    y=al_values,
    name='Answer Length',
    mode='lines+markers',
    line=dict(width=3, color='#A23B72'),
    marker=dict(size=10)
))

# Complexity Score progression
cs_values = [54.10, 59.38, 62.50]
fig.add_trace(go.Scatter(
    x=stages,
    y=cs_values,
    name='Complexity Score',
    mode='lines+markers',
    line=dict(width=3, color='#F18F01'),
    marker=dict(size=10)
))

# Baseline reference
fig.add_trace(go.Scatter(
    x=stages,
    y=[60.16, 60.16, 60.16],
    name='Baseline',
    mode='lines',
    line=dict(width=2, dash='dash', color='#2E86AB')
))

fig.update_layout(
    title='PHI-2 Performance Across Curriculum Stages',
    xaxis_title='Curriculum Stage',
    yaxis_title='Exact Match Accuracy (%)',
    yaxis_range=[48, 65],
    height=500,
    showlegend=True
)

fig.show()

#### PHI-2 with Answer Length Method
- **Easy stage:** 50.59% (dropped from 60.16% baseline!)
- **Normal stage:** 55.86% (still below baseline)
- **Difficult stage:** 59.38% (almost recovered)

**Problem:** This curriculum may have "easy" problems that are actually conceptually hard, causing catastrophic forgetting.

#### PHI-2 with Complexity Score Method
- **Easy stage:** 54.10% (smaller drop)
- **Normal stage:** 59.38% (approaching baseline)
- **Difficult stage:** 62.50% (exceeds baseline!)

**Success:** More principled difficulty estimation leads to proper progressive learning.

### Why Answer Length Failed

Answer length is a poor proxy for difficulty because:

1. **Verbose ≠ Complex:** Some problems have long explanations but simple math
2. **Terse ≠ Easy:** "What is 127 × 89?" is short but requires careful calculation
3. **Solution style varies:** Different annotators write different length solutions

Example misclassification:
```
"Mary has 3 apples. John has 5. How many total?"
→ Classified as DIFFICULT (long explanation in answer)
But conceptually: EASY (simple addition)
```

### Why Complexity Score Worked

The complexity score better captures true difficulty by considering:

1. **Number of reasoning steps** (more steps = harder)
2. **Operation types** (multiplication harder than addition)
3. **Problem structure** (multi-step vs. single-step)

This aligns with how humans perceive math difficulty.

---

## Model Size Matters

In [ ]:
# Improvement Over Baseline
curriculum_methods = ['Answer Length', 'Complexity Score']

phi2_improvements = [
    59.38 - 60.16,  # Answer Length
    62.50 - 60.16   # Complexity Score
]

smollm2_improvements = [
    2.73 - 2.15,    # Answer Length
    2.93 - 2.15     # Complexity Score
]

fig = go.Figure()

fig.add_trace(go.Bar(
    name='PHI-2 (2.7B)',
    x=curriculum_methods,
    y=phi2_improvements,
    text=[f"{v:+.2f}%" for v in phi2_improvements],
    textposition='outside',
    marker_color=['#D62246' if v < 0 else '#06A77D' for v in phi2_improvements]
))

fig.add_trace(go.Bar(
    name='SmolLM2 (135M)',
    x=curriculum_methods,
    y=smollm2_improvements,
    text=[f"{v:+.2f}%" for v in smollm2_improvements],
    textposition='outside',
    marker_color=['#D62246' if v < 0 else '#06A77D' for v in smollm2_improvements]
))

fig.update_layout(
    title='Improvement Over Baseline',
    xaxis_title='Curriculum Method',
    yaxis_title='Accuracy Change (%)',
    barmode='group',
    height=500,
    showlegend=True
)

# Add zero line
fig.add_hline(y=0, line_dash="dash", line_color="black", line_width=2)

fig.show()

### PHI-2 (2.7B): The Sweet Spot
- Has enough capacity to benefit from curriculum
- Complexity curriculum: **+2.34% improvement**
- Likely can learn from progressive examples

### SmolLM2 (135M): Too Small?
- Baseline performance: only 2.15% accuracy
- Curriculum improvement: +0.78% (marginal)
- **Hypothesis:** Model lacks fundamental reasoning capacity; curriculum can't help much

**Insight:** Curriculum learning appears most effective for medium-sized models that have sufficient capacity but can benefit from structured learning.

---

## Methodology Details

### Metrics Tracked

1. **Exact Match:** Perfect answer (strictest)
2. **Contains Answer:** Answer appears in output (lenient)
3. **Format Correct:** Uses proper GSM8K format

**Finding:** Improvements were consistent across all metrics for PHI-2 complexity curriculum, suggesting genuine reasoning improvement, not just format learning.

In [ ]:
# Method Comparison with annotations
fig = go.Figure()

phi2_all = [60.16, 59.38, 62.50]
phi2_colors = ['#2E86AB', '#D62246', '#06A77D']

fig.add_trace(go.Bar(
    name='PHI-2',
    x=methods,
    y=phi2_all,
    text=[f"{v:.2f}%" for v in phi2_all],
    textposition='outside',
    marker_color=phi2_colors,
    marker_line_color='#333',
    marker_line_width=1.5
))

smollm2_all = [2.15, 2.73, 2.93]
smollm2_colors = ['#2E86AB', '#06A77D', '#06A77D']

fig.add_trace(go.Bar(
    name='SmolLM2',
    x=methods,
    y=smollm2_all,
    text=[f"{v:.2f}%" for v in smollm2_all],
    textposition='outside',
    marker_color=smollm2_colors,
    marker_line_color='#333',
    marker_line_width=1.5
))

fig.update_layout(
    title='Final Performance Comparison by Method',
    xaxis_title='Training Method',
    yaxis_title='Exact Match Accuracy (%)',
    barmode='group',
    height=500,
    showlegend=True,
    annotations=[
        dict(x=1, y=59.38, text="-0.78%", showarrow=True, arrowhead=2, ax=0, ay=-40, font=dict(color='#D62246', size=12)),
        dict(x=2, y=62.50, text="+2.34%", showarrow=True, arrowhead=2, ax=0, ay=-40, font=dict(color='#06A77D', size=12))
    ]
)

fig.show()

---

## Implementation Notes

### The Merge-and-Continue Approach

For true curriculum learning, I merged LoRA adapters into the base model between stages:

```
Stage 1: Base Model → Train on Easy → Merge → Save
Stage 2: Merged Model → Train on Normal → Merge → Save
Stage 3: Merged Model → Train on Difficult → Final Model
```

This prevents:
- Stacking multiple LoRA adapters (wrong!)
- Starting from scratch each stage (defeats the purpose!)
- Catastrophic forgetting of previous stages

### W&B Logging Trick

To get continuous training curves across stages in Weights & Biases:

```python
cumulative_steps = 0
for stage in ['easy', 'normal', 'difficult']:
    train(stage, global_step_offset=cumulative_steps)
    cumulative_steps += steps_this_stage
```

This ensures stage 2 doesn't overwrite stage 1's logs!

---

## Practical Takeaways

### For Practitioners

1. ✅ **DO** use curriculum learning for medium-sized models (1-10B params)
2. ✅ **DO** invest time in good difficulty estimation
3. ✅ **DO** merge models between curriculum stages
4. ❌ **DON'T** use naive heuristics (like text length) for difficulty
5. ❌ **DON'T** expect curriculum to fix fundamentally weak models

### For Researchers

1. **Curriculum design is critical** - wrong curriculum can hurt performance
2. **Multi-factor difficulty scores** outperform single-factor heuristics
3. **Model capacity matters** - curriculum learning has diminishing returns for tiny models
4. **Need better difficulty estimation** - this remains an open problem

---

## Conclusion

Curriculum learning *can* improve mathematical reasoning in language models, but success hinges on intelligent curriculum design. My experiments show:

1. **A 2.34% improvement** is achievable with proper difficulty estimation
2. **Wrong curricula can hurt** - answer length reduced performance by 0.78%
3. **Model size matters** - tiny models (135M) see minimal benefit
4. **Complexity matters more than length** - multi-factor difficulty scores work better

The most surprising finding? **The wrong curriculum is worse than no curriculum at all.** This highlights the importance of careful curriculum design and suggests that more research is needed on automatic difficulty estimation for mathematical problems.

### The Bigger Picture

This work contributes to the growing evidence that **how we train** matters as much as **what we train on**. As models get larger and training becomes more expensive, techniques like curriculum learning that improve training efficiency and final performance become increasingly valuable.

The question isn't whether curriculum learning works - it's **how to design the right curriculum** for your specific task and model size.

---

## Export Results

In [ ]:
# Save results to CSV
df.to_csv('curriculum_learning_results.csv', index=False)
print("Results saved to curriculum_learning_results.csv")

# Summary statistics
print("\n=== Summary Statistics ===")
print(f"\nPHI-2 Best Method: Complexity Score (62.50%)")
print(f"PHI-2 Worst Method: Answer Length (59.38%)")
print(f"PHI-2 Improvement Range: {62.50 - 59.38:.2f}%")

print(f"\nSmolLM2 Best Method: Complexity Score (2.93%)")
print(f"SmolLM2 Worst Method: Baseline (2.15%)")
print(f"SmolLM2 Improvement Range: {2.93 - 2.15:.2f}%")

---

## References and Links

- **Blog Post:** [https://khrtim.github.io/blog/2025/11/curriculum-learning-math-reasoning/](https://khrtim.github.io/blog/2025/11/curriculum-learning-math-reasoning/)
- **GitHub Repository:** [https://github.com/KhrTim](https://github.com/KhrTim)
- **Google Scholar:** [Timur Khairulov](https://scholar.google.com/citations?user=-XrW5PAAAAAJ)

### Dataset
- GSM8K: Grade School Math 8K dataset by OpenAI

### Models
- PHI-2: Microsoft Research
- SmolLM2: HuggingFace

### Technologies Used
- PyTorch
- Hugging Face Transformers
- PEFT (Parameter-Efficient Fine-Tuning)
- Weights & Biases for experiment tracking
- Plotly for visualizations